## **집계연산**

### **PySpark 설정**

In [6]:
# [+] pyspark 불러오기
from pyspark import SparkConf, SparkContext

In [4]:
# [+] SparkConf, SparkContext 객체 생성하기
conf = SparkConf().setAppName("reduction-operations").setMaster('local')

In [5]:
sc = SparkContext(conf=conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/14 08:59:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### **reduce( ) 메서드**

In [6]:
# [+] x+y를 수행하는 reduce() 
sc.parallelize([1, 2, 3, 4, 5]).reduce(lambda x,y:x+y)

15

In [7]:
# reduce() 는 각 파티션별로 수행됨
# [+] 파티션 수: 2
rdd = sc.parallelize([1,2,3,4,5], 2)
rdd.collect()
# reduce()의 인자 함수가 교환법칙 및 결합법칙을 만족해야 한다

[1, 2, 3, 4, 5]

In [9]:
# [+] glom() : 파티션 별 RDD 값들을 배열에 모으는 변환 메서드
rdd.glom().collect()

[[1, 2], [3, 4, 5]]

In [9]:
# [+] 파티션 2개에 대해 x+y를 수행하는 reduce()
rdd.reduce(lambda x,y:x+y)

15

In [10]:
# [+] reduce() 주의점: 파티션 구성에 따라 결과 값이 달라질 수 있음
# 예: lambda x, y: (x * 2) + y
sc.parallelize([1,2,3,4,5]).reduce(lambda x,y:(x*2)+y)

57

In [11]:
# 파티션 수: 2
sc.parallelize([1,2,3,4,5], 2).reduce(lambda x,y: (x * 2) + y)

33

In [12]:
# 파티션 수: 3
sc.parallelize([1,2,3,4,5], 3).reduce(lambda x,y: (x * 2) + y)

31

In [13]:
# 파티션 수: 4
sc.parallelize([1,2,3,4,5], 4).reduce(lambda x,y: (x * 2) + y)

35

### **10000 개 정수 난수의 총합 계산하기**

In [14]:
!pip install numpy


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [15]:
# [+] numpy 라이브러리 불러오기
import numpy as np

In [16]:
# [+] [0, 100] 범위의 정수 난수 10000개 생성
x = np.random.randint(0,100, size = 10000)
x[:100]

array([ 6,  3, 35, 23, 28, 78, 94, 73, 82,  5, 78, 45, 77, 39, 67, 12, 99,
       99, 71, 70, 42, 48, 33, 24, 22,  1, 77, 40, 88, 76, 38, 62, 32, 60,
       48, 62, 59,  6, 62, 62, 59, 70, 56, 11, 61, 90, 77, 65, 92,  1, 57,
       94, 32, 77, 84, 42, 38, 60, 71,  4,  0, 27, 63,  9, 86, 51, 43,  0,
       47, 66, 83, 51, 61, 81,  3, 22, 52, 28, 72, 95, 43,  4, 15, 96, 28,
       83, 21, 63, 18, 43, 36, 38, 53, 10, 79, 94,  5, 92, 64, 25])

In [17]:
# [+] 생성한 난수 배열을 RDD로 변환(파티션 수: 8개)
rdd = sc.parallelize(x,8)

In [18]:
# 파티션 별 RDD 값 확인
partitioned_data = rdd.glom().collect()
for i, partition in enumerate(partitioned_data):
    print(f"Partition {i}: {partition}\n")
    print('\n')

[Stage 8:===================================================>       (7 + 1) / 8]

Partition 0: [np.int64(6), np.int64(3), np.int64(35), np.int64(23), np.int64(28), np.int64(78), np.int64(94), np.int64(73), np.int64(82), np.int64(5), np.int64(78), np.int64(45), np.int64(77), np.int64(39), np.int64(67), np.int64(12), np.int64(99), np.int64(99), np.int64(71), np.int64(70), np.int64(42), np.int64(48), np.int64(33), np.int64(24), np.int64(22), np.int64(1), np.int64(77), np.int64(40), np.int64(88), np.int64(76), np.int64(38), np.int64(62), np.int64(32), np.int64(60), np.int64(48), np.int64(62), np.int64(59), np.int64(6), np.int64(62), np.int64(62), np.int64(59), np.int64(70), np.int64(56), np.int64(11), np.int64(61), np.int64(90), np.int64(77), np.int64(65), np.int64(92), np.int64(1), np.int64(57), np.int64(94), np.int64(32), np.int64(77), np.int64(84), np.int64(42), np.int64(38), np.int64(60), np.int64(71), np.int64(4), np.int64(0), np.int64(27), np.int64(63), np.int64(9), np.int64(86), np.int64(51), np.int64(43), np.int64(0), np.int64(47), np.int64(66), np.int64(83), np

In [17]:
# [+] x+y 집계 연산
rdd.reduce(lambda x,y:x+y)

np.int64(496492)

### **fold( ) 메서드**
```reduce()```와 유사하며, 초기 값을 지정하여 집계를 수행

In [18]:
# [+] 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(1, lambda x,y:x+y)

12

In [19]:
# [+] 파티션 수가 2개이며, 초기 값 1이 주어지고, x+y를 수행하는 fold()
sc.parallelize([1,2,3,4],2).fold(1,lambda x,y:x+y)

13

In [20]:
# [+] 초기 값이 0이 주어지고, x*y를 수행하는 fold()
sc.parallelize([1, 2, 3, 4]).fold(0,lambda x,y:x*y)

0

### **groupBy( ) 메서드**
```groupBy()```: 인자 함수를 기준으로 그룹핑을 수행


In [19]:
# [+] 홀수/짝수로 그룹핑을 수행하는 groupBy() 코드
res = sc.parallelize([3,10,7,5,12,8,10]).groupBy(lambda x: x%2).collect()

In [20]:
# [+] groupBy() 결과 출력하기
for k, v in res: 
    print(k, list(v))

1 [3, 7, 5]
0 [10, 12, 8, 10]


### **aggregate( ) 메서드**

In [23]:
# reduce(), fold(): 입출력 RDD 형식이 같지 않으면 오류가 발생
sc.parallelize([1, 2, 3, 4])\
    .reduce(lambda x, y: (x + y, x * y)) #key value로 인식
    # single value -> pair 생성 시도할 경우 오류

26/11/14 06:52:28 ERROR Executor: Exception in task 0.0 in stage 15.0 (TID 39)
org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/rdd.py", line 1922, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/util.py", line 83, in wrapper
    return f(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_7517/2276157808.py", line 3, in <lambda>
TypeError: can only concatenate tuple (not "int") to tuple

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 15.0 failed 1 times, most recent failure: Lost task 0.0 in stage 15.0 (TID 39) (87d62242bc4b executor driver): org.apache.spark.api.python.PythonException: Traceback (most recent call last):
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1247, in main
    process()
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/worker.py", line 1239, in process
    serializer.dump_stream(out_iter, outfile)
  File "/opt/bitnami/spark/python/lib/pyspark.zip/pyspark/serializers.py", line 274, in dump_stream
    vs = list(itertools.islice(iterator, batch))
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python/pyspark/rdd.py", line 1922, in func
    yield reduce(f, iterator, initial)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/bitnami/spark/python

In [ ]:
sc.parallelize([1, 2, 3, 4]).fold((0, 0), lambda x, y: (x[0] + y, x[1] + 1))

In [ ]:
"""
    aggregate(): 입출력 형식 다른 RDD 에 대한 reduction을 수행
    - zeroValues: 파티션 초기 값
    - seqOp: 각 파티션에 대한 reduction 함수
    - combop: 파티션 결과를 합치는 reduction 함수
""" 

seqOp = (lambda x, y: (x[0] + y, x[1] + 1))
combOp = (lambda x, y: (x[0] + y[0], x[1] + y[1]))

In [ ]:
# aggregate() 수행
sc.parallelize([1, 2, 3, 4], 2).aggregate((0,0), seqOp, combOp)